# Ringkasan ekstraktif / abstraktif

Mulai ekstraktif. Model abstraktif harus sesuai task, dinilai dengan rujukan dan audit fakta/negasi; ROUGE bukan jaminan tidak berhalusinasi.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

TRAIN, TEST, SAMPLE = "train.csv", "test.csv", "sample_submission.csv"
DOC, REF, N_SENT = "document", "summary", 2
USE_T5, T5_MODEL = False, MODELS / "t5-small-sum"   # profil all


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
train, test = read_any(TRAIN), read_any(TEST)
print("train", train.shape, "| test", test.shape)
print(train.head(3).to_string())


## EDA

Rasio panjang dan tanda baca.

In [ ]:
print("rasio", round(float((train[REF].str.len() / train[DOC].str.len()).median()), 3), "| kalimat per dokumen", train[DOC].str.count(r"[.!?]").median())


## Ekstraktif

Lead-N sering sulit dikalahkan pada berita.

In [ ]:
import re
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer

def split_sents(text, window=None):
    """Tanpa tanda baca, gunakan window=30 untuk potongan kata."""
    s = [x.strip() for x in re.split(r"(?<=[.!?])\s+", str(text)) if x.strip()]
    if window and len(s) <= 1:
        w = str(text).split()
        return [" ".join(w[i:i + window]) for i in range(0, len(w), window)]
    return s

def lead_n(text, n=3):
    return " ".join(split_sents(text)[:n])

def textrank(text, n=3):
    s = split_sents(text)
    if len(s) <= n:
        return " ".join(s)
    X = TfidfVectorizer().fit_transform(s)
    S = (X @ X.T).tolil()
    S.setdiag(0)                    # tanpa self-loop: kalimat terisolasi tidak boleh menang
    r = nx.pagerank(nx.from_scipy_sparse_array(S.tocsr()), max_iter=200)
    keep = sorted(sorted(r, key=r.get, reverse=True)[:n])
    return " ".join(s[i] for i in keep)

import re
from collections import Counter

def _w(s):
    return re.findall(r"\w+", str(s).lower())

def rouge_n(pred, ref, n=1):
    g = lambda x: Counter(tuple(x[i:i + n]) for i in range(len(x) - n + 1))
    a, b = g(_w(pred)), g(_w(ref))
    m = sum((a & b).values())
    p, r = m / max(sum(a.values()), 1), m / max(sum(b.values()), 1)
    return 2 * p * r / max(p + r, 1e-9)

def rouge_l(pred, ref):
    a, b = _w(pred), _w(ref)
    d = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b):
            cur = d[j + 1]
            d[j + 1] = prev + 1 if x == y else max(d[j + 1], d[j])
            prev = cur
    p, r = d[-1] / max(len(a), 1), d[-1] / max(len(b), 1)
    return 2 * p * r / max(p + r, 1e-9)

def rouge_report(preds, refs):
    n = len(preds)
    return {"rouge1": round(sum(rouge_n(p, r) for p, r in zip(preds, refs)) / n, 4), "rouge2": round(sum(rouge_n(p, r, 2) for p, r in zip(preds, refs)) / n, 4), "rougeL": round(sum(rouge_l(p, r) for p, r in zip(preds, refs)) / n, 4)}

# ===== jalankan =====
docs, refs = train[DOC].fillna("").tolist(), train[REF].fillna("").tolist()
print("lead", rouge_report([lead_n(d, N_SENT) for d in docs], refs), "\ntextrank", rouge_report([textrank(d, N_SENT) for d in docs], refs))
pred = [lead_n(d, N_SENT) for d in test[DOC].fillna("")]


## T5

Lewati bila USE_T5=False. Konfirmasi aturan seq2seq.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

@torch.inference_mode()
def generate_texts(model, tok, texts, prefix="", max_len=384, max_new=128, beams=4, batch=8):
    dev, out = next(model.parameters()).device, []
    model.eval()
    for i in range(0, len(texts), batch):
        e = tok([prefix + str(t) for t in texts[i:i + batch]], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
        g = model.generate(**e, max_new_tokens=max_new, num_beams=beams, no_repeat_ngram_size=3)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return out

def load_seq2seq(model_path, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    return AutoModelForSeq2SeqLM.from_pretrained(str(model_path), local_files_only=True).to(device), tok

def train_seq2seq(model_path, src, tgt, prefix="", max_src=384, max_tgt=128, batch=8, epochs=1, lr=3e-4, out="run_s2s", seed=42):
    model, tok = load_seq2seq(model_path)
    data = [dict(tok(prefix + str(a), truncation=True, max_length=max_src), labels=tok(text_target=str(b), truncation=True, max_length=max_tgt)["input_ids"]) for a, b in zip(src, tgt)]
    args = Seq2SeqTrainingArguments(output_dir=out, per_device_train_batch_size=batch, num_train_epochs=epochs, learning_rate=lr, save_strategy="no",
                                    report_to="none", dataloader_num_workers=0, seed=seed, disable_tqdm=True)
    Seq2SeqTrainer(model=model, args=args, train_dataset=data, processing_class=tok, data_collator=DataCollatorForSeq2Seq(tok, model=model)).train()
    return model, tok

# ===== jalankan =====
if USE_T5:
    model, tok = load_seq2seq(T5_MODEL)
    print("t5", rouge_report(generate_texts(model, tok, docs[:200]), refs[:200]))
    pred = generate_texts(model, tok, test[DOC].fillna("").tolist())
else:
    print("USE_T5=False")


## Submission

Kolom terakhir contoh submission.

In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def encode_labels(y):
    """Kembalikan (y_int, kelas). Decode: kelas[pred]."""
    classes, yi = np.unique(np.asarray(y).astype(str), return_inverse=True)
    return yi, classes

def write_submission(sample_path, preds, out="submission.csv", pred_col=None, id_col=None, allowed=None):
    """Salin struktur contoh submission, isi prediksi, validasi, dan cetak hash."""
    s = pd.read_csv(sample_path, dtype=str)
    id_col = id_col or s.columns[0]
    pred_col = pred_col or s.columns[-1]
    assert len(s) == len(preds), f"jumlah baris {len(preds)} != contoh {len(s)}"
    s[pred_col] = preds
    assert s[pred_col].notna().all(), "ada NaN"
    assert s[id_col].is_unique, "id ganda"
    if allowed is not None:
        bad = set(s[pred_col]) - set(allowed)
        assert not bad, f"nilai asing: {bad}"
    s.to_csv(out, index=False)
    print(out, s.shape, "sha256", hashlib.sha256(Path(out).read_bytes()).hexdigest()[:16], s[pred_col].value_counts(normalize=True).head(5).round(3).to_dict())
    return s

# ===== jalankan =====
sub = write_submission(SAMPLE, pred)
